# Lab 3 - Quantum Full Adder

Simple arithmetic operations do not provide a quantum advantage, but they are excellent examples for learning quantum circuit design.

In this lab, we will study a quantum implementation of a full adder.

The circuit takes three input bits:

- q0 stores the first operand bit (A)
- q1 stores the second operand bit (B)
- q2 stores the carry input (Cin)

and produces:

- q2 stores the sum bit (S)
- q3 stores the carry output (Cout)

The circuit relies exclusively on reversible quantum gates.

<img src="fulladder.png" width="50%">

---

## Why a Quantum Full Adder?

Classical computers rely heavily on arithmetic circuits.

Although a quantum full adder does not provide a computational speed-up by itself, it introduces several important concepts:

- reversible computation,
- multi-qubit interactions,
- controlled operations,
- circuit modularity,
- reusable quantum building blocks.

These concepts are fundamental for the design of larger quantum algorithms.

---

In this lab, we will reconstruct the full-adder circuit step by step and analyse how each component contributes to the final result.

## Designing a Quantum Full Adder

Recall the following classical relations:

### XOR

$
A \oplus B
$
can be implemented using a CNOT gate.

### AND

$
A \land B
$
can be implemented using a Toffoli gate.

---

For a full adder:

### Sum

$
S = A \oplus B \oplus C_{in}
$

### Carry-Out

$
C_{out} = (A \land B) \oplus \left(C_{in} \land (A \oplus B)\right)
$

Your task is to reconstruct the quantum full-adder circuit using only:

- CNOT gates
- Toffoli (CCNOT) gates

while preserving reversibility.

In [ ]:
from qat.lang.AQASM import Program, X, CNOT, CCNOT

# Create the full adder circuit
qprog = Program()

# Allocate four qubits
# q0 = A
# q1 = B
# q2 = Cin (will become Sum)
# q3 = Cout
qbits = ...

# TODO:
# Compute the first carry contribution:
# Cout <- Cout XOR (A AND B)
# Hint: use a Toffoli gate

# TODO:
# Compute:
# q1 <- A XOR B
# Hint: use a CNOT gate

# TODO:
# Compute the second carry contribution:
# Cout <- Cout XOR (Cin AND (A XOR B))
# Hint: use a Toffoli gate

# TODO:
# Compute the final SUM bit:
# Sum <- Cin XOR (A XOR B)
# Hint: use a CNOT gate

# TODO:
# Restore q1 to recover the original value of B
# Why is this step important?

...

full_adder = qprog.to_circ()

# Display the circuit
full_adder.display()

## Analysing the Circuit

Before executing the circuit, answer the following questions:

1. Which gate computes $A \land B$ and why?

2. Which gate computes $A \oplus B$ and why?

3. Why are two carry contributions required?

    $
    A \land B
    $

    and

    $
    C_{in} \land (A \oplus B)
    $

4. The last gate restores the value of q1. Why is this operation necessary? What principle of quantum computing does it illustrate?

<details>
<summary>Show solution</summary>

1. The Toffoli gate (CCNOT) is used.
   The target qubit is flipped only when both control qubits are equal to 1.

   Truth table:

   | A | B | Output |
   |---|---|--------|
   | 0 | 0 |   0    |
   | 0 | 1 |   0    |
   | 1 | 0 |   0    |
   | 1 | 1 |   1    |

   This corresponds exactly to the Boolean AND operation.

---

2. The CNOT gate is used.
   The target qubit is flipped when the control qubit is equal to 1.

   Truth table:

   | A | B | Output |
   |---|---|--------|
   | 0 | 0 |   0    |
   | 0 | 1 |   1    |
   | 1 | 0 |   1    |
   | 1 | 1 |   0    |

   This corresponds exactly to the Boolean XOR operation.

---

3. A carry-out bit is generated in two situations:
   
   - Both input bits are equal to 1:

      A AND B

   - Exactly one input bit is equal to 1 and the carry-in is also equal to 1:

      Cin AND (A XOR B)

   The full carry-out expression is therefore:
   
   Cout = (A AND B) XOR (Cin AND (A XOR B))

   The quantum circuit computes these two contributions separately and accumulates them into q3.

---

4. After computing A XOR B, qubit q1 no longer contains its original value.
   
   To preserve reversibility, we must remove intermediate computational artefacts (often called garbage information).

   The final CNOT restores q1 to its initial value B.

   This process is called uncomputation.

   Uncomputation is a fundamental technique in quantum computing because:

      - quantum operations must be reversible,
   
      - temporary information should be removed when it is no longer needed,
   
      - many quantum algorithms rely on uncomputation to clean auxiliary qubits before measurement.

   After the restoration:
      q0 = A
      q1 = B
      q2 = Sum
      q3 = Cout

   Only the useful outputs remain.

## Preparing the Input State

By default, newly allocated qubits are initialized in the computational basis state:

$$
|0\rangle
$$

To encode a classical value of `1`, we apply an `X` gate:

$$
|0\rangle \rightarrow |1\rangle
$$

The input preparation can therefore be viewed as a small quantum circuit whose sole purpose is to initialize the desired values before computation begins.

---

For this example, we want to evaluate:

$$
1 + 1 + c_{in}
$$

with:

$$
c_{in}=1
$$

Using the qubit assignments:

- $q0 = A$
- $q1 = B$
- $q2 = C_{in}$
- $q3 = C_{out}$

the required input state is:

$$
|1110\rangle
$$

where:

- $A = 1$
- $B = 1$
- $C_{in} = 1$
- $C_{out} = 0$

---

Prepare this input state by applying the appropriate `X` gates before executing the full-adder circuit.

In [ ]:
input_prog = Program()

# Allocate 4 qubits
# q0 = A
# q1 = B
# q2 = Cin
# q3 = Cout (initially 0)
qbits = ...

# TODO:
# Prepare A = 1
...

# TODO:
# Prepare B = 1
...

# TODO:
# Prepare Cin = 1
...

input_circ = input_prog.to_circ()

input_circ.display()

## Circuit Composition

So far, we have created two independent quantum circuits:

1. An input-preparation circuit that initializes the desired values.
2. A full-adder circuit that performs the computation.

This separation is a common design pattern in quantum programming:

- state preparation,
- computation,
- measurement.

Because quantum circuits are modular, we can compose these building blocks to create a larger circuit.

In the next step, combine the input-preparation circuit with the full-adder circuit to perform the addition:

$$
1 + 1 + 1
$$

In [ ]:
# TODO: compose the input circuit with the full adder circuit
full_circuit = ...

full_circuit.display()

## Executing the Full Adder

The complete quantum circuit is now ready.

It combines:

- an input preparation stage,
- a reversible full-adder circuit.

For the chosen input:

$$
A = 1,\quad B = 1,\quad C_{in}=1
$$

the circuit should perform:

$$
1 + 1 + 1
$$

Before executing the simulation, determine the expected values of:

- the sum bit $S$,
- the carry-out bit $C_{out}$.

Run the circuit and compare the measured quantum state with your prediction.

In [ ]:
from qat.qpus import get_default_qpu

# TODO: create a job from the circuit
job = ...

# Instantiate a QPU (simulator)
qpu = get_default_qpu()

# Execute the job
result = qpu.submit(job)

for sample in result:
    print(
        "State %s: Nb %s, probability %s, amplitude %s"
        % (
            sample.state,
            sample.state.int,
            sample.probability,
            sample.amplitude,
        )
    )

## Focusing on the Output Register

The previous measurement returned the complete four-qubit state:

- q0 = A
- q1 = B
- q2 = Sum
- q3 = Carry-Out

In many applications, we are interested only in the output values produced by the computation.

We therefore measure only the output register composed of:

- q2 (Sum)
- q3 (Carry-Out)

In [ ]:
job = full_circuit.to_job(qubits=[2, 3])
result = qpu.submit(job)
for sample in result:
    print(
        f"State {sample.state}: value {sample.state.lsb_int} with probability {sample.probability}"
    )

# Discussion

For the input:

$$
A = 1,\quad B = 1,\quad C_{in}=1
$$

the classical result is:

$$
1 + 1 + 1 = 3
$$

whose binary representation is:

$$
11_2
$$

Therefore:

- S = 1
- Cout = 1

The simulator output should confirm that the full-adder circuit correctly computes this result.

---

## Key Takeaways

✅ CNOT can implement XOR operations

✅ Toffoli gates can implement AND operations

✅ Quantum arithmetic circuits rely on reversible computation

✅ Intermediate results should be uncomputed when no longer needed

✅ Complex quantum circuits can be built by composing smaller reusable circuit blocks